# Set up

In [0]:
# Check what has been already installed
%pip list 

Package                            Version
---------------------------------- -------------------------------
aiohappyeyeballs                   2.4.4
aiohttp                            3.11.10
aiohttp-retry                      2.9.1
aiosignal                          1.2.0
alembic                            1.18.4
annotated-doc                      0.0.4
annotated-types                    0.7.0
anyio                              4.7.0
argon2-cffi                        21.3.0
argon2-cffi-bindings               21.2.0
arro3-core                         0.6.5
arrow                              1.3.0
asttokens                          3.0.0
astunparse                         1.6.3
async-lru                          2.0.4
attrs                              24.3.0
azure-common                       1.1.28
azure-core                         1.37.0
azure-identity                     1.20.0
azure-mgmt-core                    1.6.0
azure-mgmt-web                     8.0.0
azure-storage-blob  

In [0]:
# Install libraries
%pip install -U -qqqq databricks-openai databricks-agents

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart the runtime
dbutils.library.restartPython()

In [0]:
 %pip show databricks-openai

Name: databricks-openai
Version: 0.15.0
Summary: Support for Databricks AI support with OpenAI
Home-page: 
Author: 
Author-email: Databricks <agent-feedback@databricks.com>
License: Apache-2.0
Location: /local_disk0/.ephemeral_nfs/envs/pythonEnv-8816dc72-ac69-4a25-ad01-7a8ea32df60c/lib/python3.12/site-packages
Requires: databricks-ai-bridge, databricks-mcp, databricks-vectorsearch, mlflow, openai, openai-agents, pydantic, unitycatalog-openai
Required-by: 


In [0]:
 %pip show databricks-agents

Name: databricks-agents
Version: 1.10.1
Summary: Mosaic AI Agent Framework SDK
Home-page: UNKNOWN
Author: Databricks
Author-email: feedback@databricks.com
License: Databricks Proprietary License
Location: /local_disk0/.ephemeral_nfs/envs/pythonEnv-8816dc72-ac69-4a25-ad01-7a8ea32df60c/lib/python3.12/site-packages
Requires: boto3, botocore, databricks-sdk, dataclasses-json, googleapis-common-protos, jinja2, mlflow-skinny, numpy, pandas, pydantic, tenacity, tiktoken, tqdm, urllib3, whenever
Required-by: 


# LLM query

#### Pick the first LLM API available in your Databricks workspace

In [0]:
from databricks.sdk import WorkspaceClient
def is_endpoint_available(endpoint_name):
  try:
    client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False
  
client = WorkspaceClient()
for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
      LLM_ENDPOINT_NAME = candidate_endpoint_name
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME"

In [0]:
LLM_ENDPOINT_NAME

'databricks-meta-llama-3-3-70b-instruct'

#### Lets do s first LLM call

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient

# Automatically log traces from LLM calls for ease of debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to talk to Databricks model serving endpoints
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()


In [0]:
# Get the response
response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )

Trace(trace_id=tr-1b242999437fe82b19131b571fec16b4)

In [0]:
# Get params from the response
response

ChatCompletion(id='chatcmpl_6e8de76a-8221-4d83-96bf-9a48c0a5ae86', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of the Czech Republic is Prague.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1778156118, model='meta-llama-3.3-70b-instruct-121024', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=10, prompt_tokens=44, total_tokens=54, completion_tokens_details=None, prompt_tokens_details=None))

#### Now lets wrap it in function

In [0]:
def run_llm(prompt):
    """
    Send a user prompt to the LLM, and return a list of LLM response messages
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """
    # Initialize an empty list to store response messages
    result_msgs = []
    # Send the user prompt to the LLM endpoint and get the response
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )
    # Extract the first message from the response
    msg = response.choices[0].message
    # Convert the message to a dictionary and add it to the result list
    result_msgs.append(msg.to_dict())

    # Return the list of response messages
    return result_msgs

In [0]:
resp = run_llm("What's the capital of Brasil ?")
for message in resp:
    print(f'{message["role"]}: {message["content"]}')

assistant: The capital of Brazil is Brasília.


Trace(trace_id=tr-415989618c588b807b1e93704f4d5390)

# LLM query with a tool  

#### Add tool

In [0]:
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
from databricks.sdk import WorkspaceClient

# Automatically log traces from LLM calls for ease of debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to talk to Databricks model serving endpoints
# We'll use this to query an LLM in our agent
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Load Databricks built-in tools (a stateless Python code interpreter tool)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"], client=client
).tools

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8816dc72-ac69-4a25-ad01-7a8ea32df60c/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
# Definuj typ vypocetniho toolu
def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function(
            "system.ai.python_exec", parameters=parameters
        )
    raise ValueError(f"Unknown tool: {tool_name}")

In [0]:

# Cela smycka Agentniho RAG systemu

def run_agent(prompt):
    """
    Send a user prompt to the LLM, and return a list of LLM response messages
    The LLM is allowed to call the code interpreter tool if needed, to respond to the user
    """

    # KROK A: První dotaz na model - "Co s tím mám dělat?"
    result_msgs = []
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
    )
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())


    # KROK B: Pokud model vyžádal nástroj, spustíme ho
    if msg.tool_calls:
        call = msg.tool_calls[0]
        tool_result = call_tool(call.function.name, json.loads(call.function.arguments))
        result_msgs.append(
            {
                "role": "tool",
                "content": tool_result.value,
                "name": call.function.name,
                "tool_call_id": call.id,
            }
        )

        # KROK C: Druhý dotaz na model - "Tady máš výsledek výpočtu, teď to lidsky ukonči"
        final_response = openai_client.chat.completions.create(
            model=LLM_ENDPOINT_NAME,
            messages=result_msgs,
        )
        
        final_msg = final_response.choices[0].message
        result_msgs.append(final_msg.to_dict())

    return result_msgs

In [0]:
import json
answer = run_agent("What is the square root of 9?")
for message in answer:
    print(f'{message["role"]}: {message["content"]}')



/local_disk0/.ephemeral_nfs/envs/pythonEnv-8816dc72-ac69-4a25-ad01-7a8ea32df60c/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


assistant: None
tool: 3.0

assistant: The code calculates the square root of 9 and prints the result. The square root of 9 is 3, so the output is 3.0.


[Trace(trace_id=tr-c6e837f3b9f7b2e4aafb15a223d1cf8e), Trace(trace_id=tr-8975b7168297221744860e183de19625)]